In [1]:
import pandas as pd
from pathlib import Path

P = Path.home() / "Downloads" / "data" / "processed"
complaints = pd.read_parquet(P / "complaints_with_labels.parquet")
recalls = pd.read_parquet(P / "recalls_cleaned.parquet")
print(len(complaints), len(recalls))

2267457 248241


In [2]:
import numpy as np

GROUP_KEYS = ["MAKETXT", "MODELTXT", "COMPONENT"]  # what one "group" is
START = "2010-01-01"    # recalls only exist from 2010
MONTHS_TO_DROP = 3      # last months are incomplete (late reports)
MIN_TOTAL = 100         # ignore groups with fewer complaints in total
HORIZON = 12            # "recall in the next 12 months?"

In [3]:
# "FUEL SYSTEM, GASOLINE:DELIVERY" -> "FUEL SYSTEM GASOLINE"
# We keep only the part before ":" so complaints and recalls use the same names.
def top_level(s):
    s = s.astype("string").str.split(":").str[0].str.upper()
    return s.str.replace(r"[^A-Z0-9]+", " ", regex=True).str.strip()

complaints["COMPONENT"] = top_level(complaints["COMPDESC"])
recalls["COMPONENT"] = top_level(recalls["COMPNAME"])
print(complaints["COMPONENT"].nunique(), "complaint components")
print(recalls["COMPONENT"].nunique(), "recall components")

54 complaint components
41 recall components


In [4]:
c = complaints.dropna(subset=GROUP_KEYS + ["COMPLAINT_DATE"])
c = c[(c["COMPONENT"] != "") & (c["COMPLAINT_DATE"] >= START)]

# last complete month (we skip the final 3 months)
last_month = (c["COMPLAINT_DATE"].max().to_period("M") - MONTHS_TO_DROP).to_timestamp()
c = c[c["COMPLAINT_DATE"] < last_month + pd.offsets.MonthBegin(1)]

c = c.assign(month=c["COMPLAINT_DATE"].dt.to_period("M").dt.to_timestamp())
counts = c.groupby(GROUP_KEYS + ["month"]).size().rename("complaints").reset_index()

# keep only groups with enough complaints
totals = counts.groupby(GROUP_KEYS)["complaints"].sum()
keep = totals[totals >= MIN_TOTAL].index
counts = counts.set_index(GROUP_KEYS).loc[keep].reset_index()

# one row for EVERY month of every group (0 when nobody complained)
months = pd.date_range(START, last_month, freq="MS")
wide = (counts.set_index(GROUP_KEYS + ["month"])["complaints"]
              .unstack("month", fill_value=0)
              .reindex(columns=months, fill_value=0))
wide.columns.name = "month"
table = wide.stack().rename("complaints").reset_index()

print("Groups:", wide.shape[0], "| Months:", len(months), "| Rows:", len(table))

Groups: 2528 | Months: 200 | Rows: 505600


In [5]:
# recall dates per group (one date per campaign, so big campaigns don't count many times)
rc = recalls.dropna(subset=GROUP_KEYS + ["RECALL_DATE"])
rc = rc[rc["COMPONENT"] != ""].drop_duplicates(GROUP_KEYS + ["CAMPNO"])
lookup = {k: np.sort(v.to_numpy(dtype="datetime64[ns]"))
          for k, v in rc.groupby(GROUP_KEYS)["RECALL_DATE"]}

# We stand at the END of each month and look 12 months ahead
table["month_end"] = table["month"] + pd.offsets.MonthEnd(0)
table["horizon_end"] = table["month_end"] + pd.DateOffset(months=HORIZON)

me = table["month_end"].values
he = table["horizon_end"].values
label = np.zeros(len(table), dtype="int8")

for key, idx in table.groupby(GROUP_KEYS, sort=False).indices.items():
    dates = lookup.get(key)
    if dates is None:
        continue                                   # this group never had a recall
    pos = np.searchsorted(dates, me[idx], side="right")   # first recall after month end
    ok = pos < len(dates)
    hit = np.zeros(len(idx), dtype=bool)
    hit[ok] = dates[pos[ok]] <= he[idx][ok]
    label[idx] = hit

table["recall_next_12m"] = label

# only keep months where the whole 12-month window is inside the recall data
recall_end = recalls["RECALL_DATE"].max()
final = table[table["horizon_end"] <= recall_end].copy()
final.to_parquet("group_month_table.parquet", index=False)

In [6]:
print("Rows:", len(final))
print("Positive rows (label 1):", int(final["recall_next_12m"].sum()),
      f"({final['recall_next_12m'].mean():.2%})")

g = final.groupby(GROUP_KEYS)["recall_next_12m"].max()
print("Groups:", len(g), "| groups with at least one positive month:", int(g.sum()))
print("Groups that appear in the recall data at all:", sum(k in lookup for k in g.index))

print(final.groupby(final["month"].dt.year)["recall_next_12m"].agg(["sum", "mean"]).round(3))

Rows: 480320
Positive rows (label 1): 38771 (8.07%)
Groups: 2528 | groups with at least one positive month: 1301
Groups that appear in the recall data at all: 1310
        sum   mean
month             
2010   1243  0.041
2011   1220  0.040
2012   1983  0.065
2013   2858  0.094
2014   2923  0.096
2015   3107  0.102
2016   2787  0.092
2017   2793  0.092
2018   2912  0.096
2019   2703  0.089
2020   2573  0.085
2021   2364  0.078
2022   2157  0.071
2023   2648  0.087
2024   2531  0.083
2025   1969  0.078


In [7]:
f = final.sort_values(GROUP_KEYS + ["month"]).copy()
g = f.groupby(GROUP_KEYS)["complaints"]

# average complaints in the last 3 / 6 / 12 months (including this month)
for w in [3, 6, 12]:
    f[f"mean_{w}m"] = g.transform(lambda s: s.rolling(w, min_periods=1).mean())

# is this month higher than the usual level? (spike measure)
f["ratio_3_vs_12"] = f["mean_3m"] / f["mean_12m"].clip(lower=0.5)

# change vs 3 months ago (is it rising?)
f["change_3m"] = f["complaints"] - g.shift(3).fillna(0)

# size of the group so far
f["total_so_far"] = g.cumsum()

FEATURES = ["complaints", "mean_3m", "mean_6m", "mean_12m",
            "ratio_3_vs_12", "change_3m", "total_so_far"]
print(f[FEATURES].describe().round(2))

       complaints    mean_3m    mean_6m   mean_12m  ratio_3_vs_12  change_3m  \
count   480320.00  480320.00  480320.00  480320.00      480320.00  480320.00   
mean         2.23       2.22       2.22       2.21           0.74       0.04   
std          6.39       5.79       5.59       5.36           0.66       5.74   
min          0.00       0.00       0.00       0.00           0.00   -1033.00   
25%          0.00       0.00       0.17       0.17           0.00      -1.00   
50%          1.00       0.67       0.83       0.83           0.67       0.00   
75%          2.00       2.33       2.17       2.17           1.16       1.00   
max       1086.00     563.50     563.50     563.50           4.00    1086.00   

       total_so_far  
count     480320.00  
mean         201.13  
std          370.01  
min            0.00  
25%           17.00  
50%           97.00  
75%          210.00  
max         6682.00  


In [9]:
# Train on 2010-2018, skip 2019 (its 12-month window overlaps the test years),
# test on 2020 onward.
train = f[f["month"] < "2019-01-01"]
test = f[f["month"] >= "2020-01-01"]
print("Train:", len(train), "positives:", train["recall_next_12m"].mean().round(3))
print("Test: ", len(test), "positives:", test["recall_next_12m"].mean().round(3))

Train: 273024 positives: 0.08
Test:  176960 positives: 0.08


In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

y_tr, y_te = train["recall_next_12m"], test["recall_next_12m"]

# Baseline: "more complaints this month = higher risk"
print("Baseline (complaint count) ROC-AUC:", round(roc_auc_score(y_te, test["complaints"]), 3),
      "| PR-AUC:", round(average_precision_score(y_te, test["complaints"]), 3))

# Logistic Regression
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
model.fit(train[FEATURES], y_tr)
p = model.predict_proba(test[FEATURES])[:, 1]
print("Logistic ROC-AUC:", round(roc_auc_score(y_te, p), 3),
      "| PR-AUC:", round(average_precision_score(y_te, p), 3))
print("Random guessing PR-AUC would be about:", round(y_te.mean(), 3))

Baseline (complaint count) ROC-AUC: 0.61 | PR-AUC: 0.125
Logistic ROC-AUC: 0.608 | PR-AUC: 0.129
Random guessing PR-AUC would be about: 0.08


In [11]:
# FIRE / CRASH are "Y" or "N"; INJURED / DEATHS are numbers (stored as text)
c2 = c.copy()
c2["fire"] = (c2["FIRE"].astype("string").str.strip().str.upper() == "Y").astype(int)
c2["crash"] = (c2["CRASH"].astype("string").str.strip().str.upper() == "Y").astype(int)
c2["injury"] = (pd.to_numeric(c2["INJURED"], errors="coerce").fillna(0) > 0).astype(int)
c2["death"] = (pd.to_numeric(c2["DEATHS"], errors="coerce").fillna(0) > 0).astype(int)

flags = (c2.groupby(GROUP_KEYS + ["month"])[["fire", "crash", "injury", "death"]]
           .sum().reset_index())

# attach to our table; months with no complaints get 0
f2 = f.merge(flags, on=GROUP_KEYS + ["month"], how="left")
f2[["fire", "crash", "injury", "death"]] = f2[["fire", "crash", "injury", "death"]].fillna(0)
f2 = f2.sort_values(GROUP_KEYS + ["month"]).reset_index(drop=True)
print(f2[["fire", "crash", "injury", "death"]].sum())

fire      20795.0
crash     53431.0
injury    31279.0
death      1128.0
dtype: float64


In [12]:
g2 = f2.groupby(GROUP_KEYS)

for col in ["fire", "crash", "injury", "death"]:
    # total in the last 3 and 12 months (including this month)
    f2[f"{col}_3m"] = g2[col].transform(lambda s: s.rolling(3, min_periods=1).sum())
    f2[f"{col}_12m"] = g2[col].transform(lambda s: s.rolling(12, min_periods=1).sum())

# shares: what fraction of recent complaints involve each safety problem
c3 = g2["complaints"].transform(lambda s: s.rolling(3, min_periods=1).sum()).clip(lower=1)
c12 = g2["complaints"].transform(lambda s: s.rolling(12, min_periods=1).sum()).clip(lower=1)
for col in ["fire", "crash", "injury", "death"]:
    f2[f"{col}_share_3m"] = f2[f"{col}_3m"] / c3
    f2[f"{col}_share_12m"] = f2[f"{col}_12m"] / c12

SAFETY = [f"{col}_{s}" for col in ["fire", "crash", "injury", "death"]
          for s in ["3m", "12m", "share_3m", "share_12m"]]
FEATURES2 = FEATURES + SAFETY
print(len(FEATURES2), "features")

23 features


In [13]:
train2 = f2[f2["month"] < "2019-01-01"]
test2 = f2[f2["month"] >= "2020-01-01"]
y_tr2, y_te2 = train2["recall_next_12m"], test2["recall_next_12m"]

# Logistic Regression with the new clues
m1 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))
m1.fit(train2[FEATURES2], y_tr2)
p1 = m1.predict_proba(test2[FEATURES2])[:, 1]
print("Logistic + safety  ROC-AUC:", round(roc_auc_score(y_te2, p1), 3),
      "| PR-AUC:", round(average_precision_score(y_te2, p1), 3))

# A stronger model, to see if it can use the clues better
from sklearn.ensemble import HistGradientBoostingClassifier
m2 = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.1, max_iter=200, random_state=0)
m2.fit(train2[FEATURES2], y_tr2)
p2 = m2.predict_proba(test2[FEATURES2])[:, 1]
print("Gradient Boosting  ROC-AUC:", round(roc_auc_score(y_te2, p2), 3),
      "| PR-AUC:", round(average_precision_score(y_te2, p2), 3))

print("\nBefore (counts only): ROC-AUC 0.608 | PR-AUC 0.129")
print("Random guessing PR-AUC about:", round(y_te2.mean(), 3))

Logistic + safety  ROC-AUC: 0.621 | PR-AUC: 0.139
Gradient Boosting  ROC-AUC: 0.608 | PR-AUC: 0.127

Before (counts only): ROC-AUC 0.608 | PR-AUC 0.129
Random guessing PR-AUC about: 0.08


In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

keys_df = f2[GROUP_KEYS].drop_duplicates()
ct = c.merge(keys_df, on=GROUP_KEYS)            # only complaints in our groups
ct = ct[ct["CDESCR"].notna()].copy()
ct["text"] = ct["CDESCR"].astype("string").str.lower()
print("Complaints with text:", len(ct))

Complaints with text: 1142226


In [15]:
train_text = ct[ct["COMPLAINT_DATE"] < "2019-01-01"]
sample = train_text["text"].sample(200_000, random_state=0)

tfidf = TfidfVectorizer(max_features=20000, min_df=20, stop_words="english",
                        sublinear_tf=True, dtype=np.float32)
tfidf.fit(sample)

svd = TruncatedSVD(n_components=50, random_state=0)
svd.fit(tfidf.transform(sample))
print("Done. Topics explain", round(svd.explained_variance_ratio_.sum(), 2), "of the word patterns")

Done. Topics explain 0.16 of the word patterns


In [16]:
parts = []
for i in range(0, len(ct), 100_000):
    X = tfidf.transform(ct["text"].iloc[i:i+100_000])
    parts.append(svd.transform(X).astype(np.float32))
    print(i, end=" ")
V = np.vstack(parts)
print("\nShape:", V.shape)

0 100000 200000 300000 400000 500000 600000 700000 800000 900000 1000000 1100000 
Shape: (1142226, 50)


In [17]:
SVD_COLS = [f"svd{i}" for i in range(50)]
vec = pd.concat([ct[GROUP_KEYS + ["month"]].reset_index(drop=True),
                 pd.DataFrame(V, columns=SVD_COLS)], axis=1)
sums = vec.groupby(GROUP_KEYS + ["month"])[SVD_COLS].sum().reset_index()

f3 = f2.merge(sums, on=GROUP_KEYS + ["month"], how="left")
f3[SVD_COLS] = f3[SVD_COLS].fillna(0)
f3 = f3.sort_values(GROUP_KEYS + ["month"]).reset_index(drop=True)

g3 = f3.groupby(GROUP_KEYS)
n3 = g3["complaints"].transform(lambda s: s.rolling(3, min_periods=1).sum()).clip(lower=1)
for col in SVD_COLS:
    f3[col + "_3m"] = g3[col].transform(lambda s: s.rolling(3, min_periods=1).sum()) / n3

TEXT = [col + "_3m" for col in SVD_COLS]
FEATURES3 = FEATURES2 + TEXT
print(len(FEATURES3), "features")

C:\Users\DELL\AppData\Local\Temp\ipykernel_19264\3125688189.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  f3[col + "_3m"] = g3[col].transform(lambda s: s.rolling(3, min_periods=1).sum()) / n3
C:\Users\DELL\AppData\Local\Temp\ipykernel_19264\3125688189.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  f3[col + "_3m"] = g3[col].transform(lambda s: s.rolling(3, min_periods=1).sum()) / n3
C:\Users\DELL\AppData\Local\Temp\ipykernel_19264\3125688189.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually 

73 features


C:\Users\DELL\AppData\Local\Temp\ipykernel_19264\3125688189.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  f3[col + "_3m"] = g3[col].transform(lambda s: s.rolling(3, min_periods=1).sum()) / n3


In [18]:
train3 = f3[f3["month"] < "2019-01-01"]
test3 = f3[f3["month"] >= "2020-01-01"]
y_tr3, y_te3 = train3["recall_next_12m"], test3["recall_next_12m"]

m3 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight="balanced"))
m3.fit(train3[FEATURES3], y_tr3)
p3 = m3.predict_proba(test3[FEATURES3])[:, 1]
print("Logistic + safety + text  ROC-AUC:", round(roc_auc_score(y_te3, p3), 3),
      "| PR-AUC:", round(average_precision_score(y_te3, p3), 3))

m4 = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.1, max_iter=200, random_state=0)
m4.fit(train3[FEATURES3], y_tr3)
p4 = m4.predict_proba(test3[FEATURES3])[:, 1]
print("Gradient Boosting + text  ROC-AUC:", round(roc_auc_score(y_te3, p4), 3),
      "| PR-AUC:", round(average_precision_score(y_te3, p4), 3))

print("\nBefore: counts 0.129 | + safety 0.139 | random 0.08 (PR-AUC)")

Logistic + safety + text  ROC-AUC: 0.668 | PR-AUC: 0.154
Gradient Boosting + text  ROC-AUC: 0.662 | PR-AUC: 0.166

Before: counts 0.129 | + safety 0.139 | random 0.08 (PR-AUC)
